# Q13a — Qwen1.5-7B: Layer-by-Layer Constellation Analysis

**Mirrors:** `13a_[Analysis]_Layer_by_Layer_Constellation_Analysis.ipynb` (LLaMA)

**Goal:** Test whether Qwen1.5-7B shows the same task-identity constellation structure.
- Claim 1: Inter-task silhouette peaks at an early mid-layer
- Galaxy PCA at the constellation peak
- Within-task behavioral crystallisation silhouette
- Per-task centroid distance (CDist) vs inter-task reference

**Qwen architecture:** 31 layers, 4096-dim residual stream  
**Known result from NB20:** constellation peak at L5 (silhouette=0.458)  
**Key limitation:** OR=22, RH=1 — per-task directional analysis is limited.

**No GPU required** — works from pre-computed embeddings.

In [ ]:
# ── PATH CONFIGURATION ────────────────────────────────────────────────────────
# Set EMBEDDINGS_DIR to wherever the Qwen embeddings are stored.
#
# Option A — Google Colab (run the cell below to copy from Drive first):
#   EMBEDDINGS_DIR = './embeddings_qwen'
#
# Option B — Local machine (downloaded from Drive):
#   EMBEDDINGS_DIR = '/path/to/embeddings_qwen'
#
# The directory must contain:
#   - one .csv file (with a 'torch_path', 'llm_evaluation', 'refusal_class' column)
#   - the corresponding .pt file referenced by torch_path

EMBEDDINGS_DIR = './embeddings_qwen'
FIGURES_DIR    = './figures'

# Qwen architecture constants
N_LAYERS_QWEN  = 31   # layers 0-30
KNOWN_PEAK_L   = 5    # from NB20; confirmed below

import os
os.makedirs(FIGURES_DIR, exist_ok=True)
print('Config set. EMBEDDINGS_DIR:', EMBEDDINGS_DIR)

In [ ]:
# ── COLAB ONLY: mount Drive and copy embeddings ───────────────────────────────
# Skip this cell if running locally.
# Uncomment and adjust the Drive path to match your layout.

# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
# !cp -a "/content/drive/MyDrive/embeddings/overalign_eval_qwen_1_5/." {EMBEDDINGS_DIR}/.
print('(Colab cell skipped — running locally)')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA as PCA_sk
import warnings, json

warnings.filterwarnings('ignore')

# ACL publication style (matches LLaMA notebooks)
plt.rcParams.update({
    'font.size': 12, 'font.family': 'serif',
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.labelsize': 12, 'legend.fontsize': 10,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.25,
    'lines.linewidth': 2.0,
})

# Canonical task colours (identical across all notebooks)
TASK_COLORS = {
    'sentiment_analysis': '#E74C3C',
    'translate':          '#3498DB',
    'rephrase':           '#F39C12',
    'rag_qa':             '#27AE60',
    'cryptanalysis':      '#9B59B6',
}

PAL = {
    'inter': '#2C3E50',
    'behav': '#E74C3C',
    'ref':   '#7F8C8D',
}

BENIGN_TASKS = ['sentiment_analysis', 'translate', 'cryptanalysis', 'rag_qa']

print('✓ Libraries loaded')

In [ ]:
# ── Load Qwen embeddings ──────────────────────────────────────────────────────
def to_np(d):
    return {
        k: (v.float().numpy().astype(np.float32) if isinstance(v, torch.Tensor)
            else np.array([e.float().numpy().astype(np.float32) for e in v]))
        for k, v in d.items()
    }

csv_files = sorted(f for f in os.listdir(EMBEDDINGS_DIR) if f.endswith('.csv'))
csv_df = pd.read_csv(os.path.join(EMBEDDINGS_DIR, csv_files[-1]))

torch_path = csv_df['torch_path'].iloc[0]
if not os.path.isabs(torch_path):
    torch_path = os.path.join(EMBEDDINGS_DIR, os.path.basename(torch_path))
torch_data = torch.load(torch_path, map_location='cpu')

embeddings_np        = to_np(torch_data['embeddings'])
texts                = torch_data['texts']
text_type_labels     = np.array(torch_data['text_type_labels'])
intended_task_labels = np.array(torch_data['intended_task_labels'])
response_labels      = csv_df['llm_evaluation'].values
refusal_labels       = csv_df['refusal_class'].values

# Qwen has 31 layers (layer_0 to layer_30)
LAYER_NAMES = sorted(
    [k for k in embeddings_np if 'input_norm' in k],
    key=lambda x: int(x.split('layer_')[1].split('_')[0])
)
LAYER_NUMS  = [int(ln.split('layer_')[1].split('_')[0]) for ln in LAYER_NAMES]
ALL_TASKS   = sorted(np.unique(intended_task_labels).tolist())
N_SAMPLES   = len(texts)

print(f'Loaded: {N_SAMPLES} samples | {len(LAYER_NAMES)} layers | dim={list(embeddings_np.values())[0].shape[1]}')
print(f'Tasks: {ALL_TASKS}')

In [ ]:
# ── Canonical masks (identical to NB13a) ─────────────────────────────────────
refusing_mask = (refusal_labels == 'direct_refusal') | (refusal_labels == 'indirect_refusal')
answered_mask = refusal_labels == 'direct_answer'
harmful_mask  = text_type_labels == 'harmful_instruction'
benign_mask   = np.isin(intended_task_labels, BENIGN_TASKS)

OVER_REFUSAL_MASK      = benign_mask  & refusing_mask
REFUSED_HARMFUL_MASK   = harmful_mask & refusing_mask
HARMLESS_ANSWERED_MASK = benign_mask  & answered_mask
TARGET_MASK = (
    ((response_labels == 'cautious') | (response_labels == 'not_harmful')) & answered_mask
)
binary_refusal = np.where(refusing_mask, 'refused', 'answered')

beh_labels = np.full(N_SAMPLES, 'other', dtype=object)
beh_labels[REFUSED_HARMFUL_MASK]   = 'refused_harmful'
beh_labels[OVER_REFUSAL_MASK]      = 'over_refusal'
beh_labels[HARMLESS_ANSWERED_MASK] = 'harmless_answered'

TASKS_WITH_OR = [
    t for t in ALL_TASKS
    if ((intended_task_labels == t) & OVER_REFUSAL_MASK).sum() >= 3
]

print('=== MASK COUNTS ===')
print(f'  Over-refusal (OR):         {OVER_REFUSAL_MASK.sum():>4d}')
print(f'  Refused-harmful (RH):      {REFUSED_HARMFUL_MASK.sum():>4d}')
print(f'  Harmless-answered (HA):    {HARMLESS_ANSWERED_MASK.sum():>4d}')
print(f'  Target (cautious/answered):{TARGET_MASK.sum():>4d}')
print(f'  Tasks with ≥3 OR samples:  {TASKS_WITH_OR}')
print()
if REFUSED_HARMFUL_MASK.sum() < 5:
    print('⚠ RH n < 5 — per-task harmful-refusal directions and type probe unavailable.')
print()
print(f'  {"Task":<22} {"Total":>6} {"OR":>5} {"RH":>5} {"HA":>6}')
for task in ALL_TASKS:
    tm = intended_task_labels == task
    print(f'  {task:<22} {tm.sum():>6}'
          f' {(tm & OVER_REFUSAL_MASK).sum():>5}'
          f' {(tm & REFUSED_HARMFUL_MASK).sum():>5}'
          f' {(tm & HARMLESS_ANSWERED_MASK).sum():>6}')

In [ ]:
# ── Pre-compute all layer metrics ─────────────────────────────────────────────
print('Pre-computing metrics across all layers...\n')

pca2_per_layer     = {}
emb2d_per_layer    = {}
task_sil_per_layer = {}
beh_sil_per_layer  = {}
inter_centroid_dist= {}
intra_task_spread  = {}
task_centroids_all = {}
cryst_sil          = {}

or_ha_mask    = OVER_REFUSAL_MASK | HARMLESS_ANSWERED_MASK
beh_sil_lbls  = np.where(OVER_REFUSAL_MASK[or_ha_mask], 'or', 'ha')

for i, lname in enumerate(LAYER_NAMES):
    emb = embeddings_np[lname]

    # PCA-2D
    pca = PCA_sk(n_components=2, random_state=42)
    emb2d_per_layer[lname] = pca.fit_transform(emb)
    pca2_per_layer[lname]  = pca

    # Inter-task silhouette (cosine distance, full-dim)
    try:
        task_sil_per_layer[lname] = float(
            silhouette_score(emb, intended_task_labels, metric='cosine'))
    except Exception:
        task_sil_per_layer[lname] = np.nan

    # Behavioral silhouette (OR vs HA)
    if or_ha_mask.sum() >= 4 and len(set(beh_sil_lbls)) == 2:
        try:
            beh_sil_per_layer[lname] = float(
                silhouette_score(emb[or_ha_mask], beh_sil_lbls, metric='cosine'))
        except Exception:
            beh_sil_per_layer[lname] = np.nan
    else:
        beh_sil_per_layer[lname] = np.nan

    # Inter-centroid distance
    cents = {t: emb[intended_task_labels == t].mean(0) for t in ALL_TASKS}
    task_centroids_all[lname] = cents
    pairwise = [float(np.linalg.norm(cents[ti] - cents[tj]))
                for ii, ti in enumerate(ALL_TASKS)
                for jj, tj in enumerate(ALL_TASKS) if ii < jj]
    inter_centroid_dist[lname] = float(np.mean(pairwise))

    spreads = []
    for t in ALL_TASKS:
        t_emb = emb[intended_task_labels == t]
        spreads.extend(np.linalg.norm(t_emb - cents[t], axis=1).tolist())
    intra_task_spread[lname] = float(np.mean(spreads))

    # Within-task crystallization silhouette
    for task in ALL_TASKS:
        mask  = intended_task_labels == task
        t_bin = binary_refusal[mask]
        if (t_bin == 'refused').sum() < 2 or (t_bin == 'answered').sum() < 2:
            cryst_sil[(task, lname)] = np.nan
            continue
        try:
            cryst_sil[(task, lname)] = float(
                silhouette_score(emb[mask], t_bin, metric='euclidean'))
        except Exception:
            cryst_sil[(task, lname)] = np.nan

    if i % 8 == 0:
        print(f'  L{i:02d}: inter-task sil={task_sil_per_layer[lname]:.3f}')

sil_vals  = [task_sil_per_layer[l]  for l in LAYER_NAMES]
beh_vals  = [beh_sil_per_layer[l]   for l in LAYER_NAMES]
inter_vals= [inter_centroid_dist[l]  for l in LAYER_NAMES]
intra_vals= [intra_task_spread[l]    for l in LAYER_NAMES]

peak_inter = int(np.nanargmax(sil_vals))
peak_beh   = int(np.nanargmax(beh_vals))
PEAK_LAYER = LAYER_NUMS[peak_inter]
PEAK_LNAME = LAYER_NAMES[peak_inter]

print(f'\n=== KEY NUMBERS ===')
print(f'  Inter-task silhouette peak: {sil_vals[peak_inter]:.4f} at L{PEAK_LAYER}')
print(f'  Behavioral silhouette peak: {beh_vals[peak_beh]:.4f} at L{LAYER_NUMS[peak_beh]}')
print(f'  (LLaMA reference: inter=0.454@L12, behav=0.170@L22)')
print('\nAll pre-computations complete.')

In [ ]:
# ── Figure 1: Silhouette curves (inter-task + behavioral) ─────────────────────
# Mirrors: fig13a_02_silhouette_curves (LLaMA)

EARLY_BAND = (0, 4);  MID_BAND = (5, 19);  LATE_BAND = (20, 30)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 3.5))
plt.subplots_adjust(wspace=0.35, top=0.82, bottom=0.14, left=0.09, right=0.97)

for ax in (ax1, ax2):
    ax.axvspan(*EARLY_BAND, alpha=0.06, color='#95A5A6', zorder=0)
    ax.axvspan(*MID_BAND,   alpha=0.08, color='#2980B9', zorder=0)
    ax.axvspan(*LATE_BAND,  alpha=0.07, color='#E74C3C', zorder=0)

# Left: inter-task silhouette
ax1.plot(LAYER_NUMS, sil_vals, 'o-', color=PAL['inter'], lw=2.2, zorder=3)
ax1.fill_between(LAYER_NUMS, 0, sil_vals, alpha=0.12, color=PAL['inter'])
ax1.axvline(PEAK_LAYER, color=PAL['inter'], lw=1.5, ls='--', zorder=4,
            label=f'Peak L{PEAK_LAYER} ({sil_vals[peak_inter]:.3f})')
ax1.set_xlabel('Layer'); ax1.set_ylabel('Silhouette score (cosine)')
ax1.set_title('Inter-task constellation\nquality per layer', pad=4)
ax1.legend(fontsize=9); ax1.set_ylim(bottom=-0.05)
ax1.text(1, -0.04, 'early', fontsize=7, color='#95A5A6')
ax1.text(9, -0.04, 'mid', fontsize=7, color='#2980B9')
ax1.text(24, -0.04, 'late', fontsize=7, color='#E74C3C')

# Right: behavioral silhouette
beh_not_nan = [b if not np.isnan(b) else 0 for b in beh_vals]
ax2.plot(LAYER_NUMS, beh_vals, 's--', color=PAL['behav'], lw=2.2, zorder=3)
ax2.fill_between(LAYER_NUMS, 0, beh_not_nan, alpha=0.12, color=PAL['behav'])
if not np.all(np.isnan(beh_vals)):
    ax2.axvline(LAYER_NUMS[peak_beh], color=PAL['behav'], lw=1.5, ls='--', zorder=4,
                label=f'Peak L{LAYER_NUMS[peak_beh]} ({beh_vals[peak_beh]:.3f})')
    ax2.legend(fontsize=9)
else:
    ax2.text(0.3, 0.5, 'Insufficient OR/HA samples', transform=ax2.transAxes,
             ha='center', color='grey', fontsize=10)
ax2.set_xlabel('Layer'); ax2.set_ylabel('Silhouette score (cosine)')
ax2.set_title('Within-task behavioral\nseparation (OR vs HA)', pad=4)
ax2.set_ylim(bottom=-0.05)

ratio = sil_vals[peak_inter] / beh_vals[peak_beh] if not np.isnan(beh_vals[peak_beh]) else float('nan')
fig.suptitle(f'Qwen1.5-7B — Task constellation structure (peak L{PEAK_LAYER})\n'
             f'Inter-task peak: {sil_vals[peak_inter]:.3f}  |  '
             f'Behavioral peak: {beh_vals[peak_beh]:.3f}  |  '
             f'Gap ratio: {ratio:.1f}×  (LLaMA ref: 2.7×)',
             fontsize=10, fontweight='bold')

plt.savefig(f'{FIGURES_DIR}/q_fig13a_02_silhouette_curves.pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{FIGURES_DIR}/q_fig13a_02_silhouette_curves.png', bbox_inches='tight', dpi=300)
plt.show()

print('=== FIGURE DATA (Silhouette curves) ===')
print(f'  Inter-task peak: {sil_vals[peak_inter]:.4f} at L{PEAK_LAYER}')
print(f'  Behavioral peak: {beh_vals[peak_beh]:.4f} at L{LAYER_NUMS[peak_beh]}')
print(f'  Gap ratio: {ratio:.2f}× (LLaMA: 2.7×)')
print(f'  LLaMA reference: inter=0.454@L12, behav=0.170@L22')

In [ ]:
# ── Figure 2: PCA Galaxy at constellation peak ────────────────────────────────
# Mirrors: fig_nb13a_dual_galaxy (LLaMA)
# Left panel: task identity coloured; Right panel: behavior coloured

lname = PEAK_LNAME
emb2d = emb2d_per_layer[lname]
pca2  = pca2_per_layer[lname]

BEH_COLORS = {
    'harmless_answered': '#27AE60',
    'over_refusal':      '#E74C3C',
    'refused_harmful':   '#2C3E50',
    'other':             '#BDC3C7',
}

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(7.0, 4.0))
plt.subplots_adjust(wspace=0.35, top=0.88, bottom=0.12, left=0.07, right=0.97)

# --- Left: task colouring ---
for task in ALL_TASKS:
    m = intended_task_labels == task
    ax_l.scatter(emb2d[m, 0], emb2d[m, 1],
                 c=TASK_COLORS[task], s=20, alpha=0.65,
                 edgecolors='white', linewidths=0.3, zorder=2, label=task.replace('_', ' '))
    # OR triangles
    m_or = m & OVER_REFUSAL_MASK
    if m_or.any():
        ax_l.scatter(emb2d[m_or, 0], emb2d[m_or, 1],
                     c=TASK_COLORS[task], s=70, alpha=1.0, marker='^',
                     edgecolors='black', linewidths=0.7, zorder=3)
    # Centroids
    cx, cy = emb2d[m, 0].mean(), emb2d[m, 1].mean()
    ax_l.annotate(task.replace('_', '\n').replace('sentiment\nanalysis', 'sentiment'),
                  (cx, cy), fontsize=7.5, fontweight='bold', ha='center', va='center',
                  color=TASK_COLORS[task],
                  bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7, ec=TASK_COLORS[task], lw=0.8),
                  zorder=5)

var = pca2.explained_variance_ratio_
ax_l.set_xlabel(f'PC1 ({var[0]:.1%})', fontsize=9)
ax_l.set_ylabel(f'PC2 ({var[1]:.1%})', fontsize=9)
ax_l.set_title(f'Task identity — L{PEAK_LAYER}\n(silhouette={sil_vals[peak_inter]:.3f})', fontsize=10, pad=4)
ax_l.tick_params(labelsize=7)

# Task legend
handles = [mpatches.Patch(color=TASK_COLORS[t], label=t.replace('_',' ')) for t in ALL_TASKS]
handles += [Line2D([0],[0], marker='^', color='grey', lw=0, ms=7,
                   markeredgecolor='black', markeredgewidth=0.7, label='Over-refusal (▲)')]
ax_l.legend(handles=handles, loc='upper left', fontsize=7.5,
            framealpha=1.0, edgecolor='0.7')

# --- Right: behavior coloring ---
BEH_ORDER = ['harmless_answered', 'over_refusal', 'refused_harmful', 'other']
BEH_MARKERS = {'harmless_answered': 'o', 'over_refusal': '^', 'refused_harmful': 'X', 'other': 'o'}
BEH_SIZES   = {'harmless_answered': 18,  'over_refusal': 60,  'refused_harmful': 65,  'other': 8}
BEH_ALPHA   = {'harmless_answered': 0.55,'over_refusal': 0.95,'refused_harmful': 1.00, 'other': 0.12}
BEH_LABELS  = {
    'harmless_answered': f'Harmless answered (n={HARMLESS_ANSWERED_MASK.sum()})',
    'over_refusal':      f'Over-refusal (n={OVER_REFUSAL_MASK.sum()})',
    'refused_harmful':   f'Refused harmful (n={REFUSED_HARMFUL_MASK.sum()})',
    'other':             'Other',
}

for beh in BEH_ORDER:
    m = beh_labels == beh
    if not m.any():
        continue
    ax_r.scatter(emb2d[m, 0], emb2d[m, 1],
                 c=BEH_COLORS[beh], s=BEH_SIZES[beh], alpha=BEH_ALPHA[beh],
                 marker=BEH_MARKERS[beh],
                 edgecolors='black' if beh in ('over_refusal','refused_harmful') else 'white',
                 linewidths=0.7 if beh in ('over_refusal','refused_harmful') else 0.3,
                 zorder=3 if beh in ('over_refusal','refused_harmful') else 2,
                 label=BEH_LABELS[beh])

ax_r.set_xlabel(f'PC1 ({var[0]:.1%})', fontsize=9)
ax_r.set_ylabel(f'PC2 ({var[1]:.1%})', fontsize=9)
ax_r.set_title(f'Behavior — L{PEAK_LAYER}\n(OR sits inside task clusters)', fontsize=10, pad=4)
ax_r.tick_params(labelsize=7)
ax_r.legend(loc='upper right', fontsize=7.5, framealpha=1.0, edgecolor='0.7')

fig.suptitle(f'Qwen1.5-7B — 2D PCA at peak constellation layer (L{PEAK_LAYER})',
             fontsize=11, fontweight='bold')

plt.savefig(f'{FIGURES_DIR}/q_fig13a_dual_galaxy.pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{FIGURES_DIR}/q_fig13a_dual_galaxy.png', bbox_inches='tight', dpi=300)
plt.show()
print(f'✓ Saved: q_fig13a_dual_galaxy.pdf/png')

In [ ]:
# ── Figure 3: Per-task centroid distance (OR vs target) vs inter-task reference
# Mirrors: fig_nb7_cluster_separation (LLaMA)

cdist_per_task = {task: [] for task in TASKS_WITH_OR}
sil_per_task   = {task: [] for task in TASKS_WITH_OR}

for lname in LAYER_NAMES:
    emb = embeddings_np[lname]
    for task in TASKS_WITH_OR:
        m_or  = (intended_task_labels == task) & OVER_REFUSAL_MASK
        m_tgt = (intended_task_labels == task) & TARGET_MASK
        if m_or.sum() < 2 or m_tgt.sum() < 2:
            cdist_per_task[task].append(np.nan)
            sil_per_task[task].append(np.nan)
            continue
        c_or  = emb[m_or].mean(0)
        c_tgt = emb[m_tgt].mean(0)
        cdist_per_task[task].append(float(np.linalg.norm(c_or - c_tgt)))
        # Within-task behavioral silhouette
        all_m = m_or | m_tgt
        lbls  = np.where(m_or[all_m], 'or', 'tgt')
        try:
            sil_per_task[task].append(float(
                silhouette_score(emb[all_m], lbls, metric='euclidean')))
        except Exception:
            sil_per_task[task].append(np.nan)

# Inter-task reference (mean centroid distance between tasks)
inter_ref = [inter_centroid_dist[l] for l in LAYER_NAMES]

if not TASKS_WITH_OR:
    print('⚠ No tasks with ≥3 OR samples — CDist plot skipped.')
else:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.0, 5.0), sharex=True)
    plt.subplots_adjust(hspace=0.30, top=0.85, bottom=0.10, left=0.10, right=0.97)

    EARLY_BAND = (0, 4);  MID_BAND = (5, 19);  LATE_BAND = (20, 30)
    for ax in (ax1, ax2):
        ax.axvspan(*EARLY_BAND, alpha=0.06, color='#95A5A6', zorder=0)
        ax.axvspan(*MID_BAND,   alpha=0.08, color='#2980B9', zorder=0)
        ax.axvspan(*LATE_BAND,  alpha=0.07, color='#E74C3C', zorder=0)

    # Top: CDist
    ax1.plot(LAYER_NUMS, inter_ref, '--', color='#7F8C8D', lw=2.0,
             label='Mean inter-task centroid dist (reference)', zorder=2)
    ax1.fill_between(LAYER_NUMS, 0, inter_ref, alpha=0.08, color='#7F8C8D')

    for task in TASKS_WITH_OR:
        ax1.plot(LAYER_NUMS, cdist_per_task[task], '-', color=TASK_COLORS[task], lw=2.2,
                 label=f'{task.replace("_"," ")} OR↔target', zorder=3)

    ax1.axvline(PEAK_LAYER, color='#7F8C8D', lw=1.2, ls=':', zorder=4)
    ax1.set_ylabel('L2 centroid distance')
    ax1.set_title('Per-task within-cluster behavioral separation (Qwen1.5-7B)', pad=4)
    ax1.legend(fontsize=9, loc='upper left')

    # Bottom: silhouette per task
    for task in TASKS_WITH_OR:
        ax2.plot(LAYER_NUMS, sil_per_task[task], '-', color=TASK_COLORS[task], lw=2.2,
                 label=f'{task.replace("_"," ")}', zorder=3)
    ax2.axhline(0, color='grey', lw=0.8, ls=':')
    ax2.axvline(PEAK_LAYER, color='#7F8C8D', lw=1.2, ls=':', zorder=4)
    ax2.set_xlabel('Layer'); ax2.set_ylabel('Within-task silhouette')
    ax2.set_title('Within-task silhouette (OR vs answered per task)', pad=4)
    ax2.legend(fontsize=9, loc='upper left')

    plt.savefig(f'{FIGURES_DIR}/q_fig_cluster_separation.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(f'{FIGURES_DIR}/q_fig_cluster_separation.png', bbox_inches='tight', dpi=300)
    plt.show()

    print('=== FIGURE DATA (CDist) ===')
    peak_i = np.nanargmax(inter_ref)
    print(f'  Inter-task ref peak: {inter_ref[peak_i]:.2f} at L{LAYER_NUMS[peak_i]}')
    for task in TASKS_WITH_OR:
        cd = np.array(cdist_per_task[task])
        pi = np.nanargmax(cd)
        print(f'  {task}: CDist peak={cd[pi]:.2f} at L{LAYER_NUMS[pi]}')
    print(f'  LLaMA ref: behavioral gap ≈9 units vs inter-task ≈22 at peak')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 65)
print('Q13a — QWEN CONSTELLATION SUMMARY')
print('=' * 65)
print(f'Model: Qwen1.5-7B ({len(LAYER_NAMES)} layers, dim=4096)')
print(f'Constellation peak: L{PEAK_LAYER} (silhouette={sil_vals[peak_inter]:.4f})')
print(f'  LLaMA reference:  L12  (silhouette=0.4540)')
print()
if not np.isnan(beh_vals[peak_beh]):
    print(f'Behavioral silhouette peak: L{LAYER_NUMS[peak_beh]} ({beh_vals[peak_beh]:.4f})')
    print(f'  LLaMA reference: L22 (0.1700) — 2.7× gap')
    print(f'  Qwen gap: {sil_vals[peak_inter] / beh_vals[peak_beh]:.2f}×')
print()
print(f'OR samples: {OVER_REFUSAL_MASK.sum()} | RH samples: {REFUSED_HARMFUL_MASK.sum()}')
print(f'Tasks with OR≥3: {TASKS_WITH_OR}')
print('=' * 65)